# Cruise Control

This notebook demonstrates the **dynamicalnodes** framework end-to-end.
Every component — reference generator, PID controller, car plant, and Kalman filter —
is modeled as a `DynamicalSystem` and composed together in a closed-loop simulation.

## System Overview

$$
\underbrace{r_k}_{\text{reference}}
\xrightarrow{} \underbrace{u_k}_{\text{PID}}
\xrightarrow{} \underbrace{y_k}_{\text{car}}
\xrightarrow{} \underbrace{\hat{x}_k}_{\text{KF}}
\xrightarrow{} \text{(back to PID)}
$$

The car velocity tracks a half-sine reference signal.
A Kalman filter estimates the car state from noisy velocity measurements
and feeds the estimate back to the PID controller.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dynamicalnodes import DynamicalSystem

## Component Definitions

Each component is a `DynamicalSystem(f=..., h=...)` where:
- `f(x_k, ...)` → `x_{k+1}` updates the internal state
- `h(x_k, ...)` → `y_k` produces the observable output

Calling `block.step(**kwargs)` returns `(x_{k+1}, y_k)` for stateful systems.

### Reference Signal

Half-sine wave: $r_k = A \sin\!\left(\frac{\pi t_k}{T}\right)$ for $t_k \le T$, else $0$.

In [ ]:
def ref_f(rk, tk, A, T):
    return A * np.sin(np.pi * tk / T) if tk <= T else 0.0

def ref_h(rk):
    return rk

ref_block = DynamicalSystem(f=ref_f, h=ref_h)

### PID Controller

State: $(e_{k-1},\, \int e)$ — previous error and accumulated integral.

$$u_k = K_P e_k + K_I \sum e_k \Delta t + K_D \frac{e_k - e_{k-1}}{\Delta t}$$

In [ ]:
def pid_f(ck, rk, xhatk, KP, KI, KD, dt):
    """Update PID state: returns (prev_error, integral)."""
    e_prev, e_int = ck
    ek = rk - xhatk
    return (ek, e_int + ek * dt)

def pid_h(ck, rk, xhatk, KP, KI, KD, dt):
    """Compute control output from current PID state."""
    e_prev, e_int = ck
    ek = rk - xhatk
    e_der = (ek - e_prev) / dt
    return KP * ek + KI * e_int + KD * e_der

pid_block = DynamicalSystem(f=pid_f, h=pid_h)

### Car Plant

State: $p_k = [\text{position},\; \text{velocity}]^\top$.

$$
\begin{bmatrix} p_{k+1} \\ v_{k+1} \end{bmatrix}
=
\begin{bmatrix} p_k + \Delta t\, v_k \\ v_k + \Delta t\left(-\tfrac{b}{m} v_k + \tfrac{u_k}{m}\right) \end{bmatrix}
$$

The observation $y_k = v_k$ is the current velocity (plus measurement noise in the KF).

In [ ]:
def plant_f(pk, uk, m, b, dt):
    p, v = pk
    return np.array([p + dt * v,
                     v + dt * (-b / m * v + uk / m)])

def plant_h(pk):
    return pk[1]  # velocity

plant_block = DynamicalSystem(f=plant_f, h=plant_h)

### Kalman Filter

State: $z_k = (\hat{x}_k,\, P_k)$ — state estimate and covariance.

**Predict:**
$$\hat{x}_{k|k-1} = F_k \hat{x}_{k-1} + B_k u_k, \qquad P_{k|k-1} = F_k P_{k-1} F_k^\top + Q_k$$

**Update:**
$$K_k = P_{k|k-1} H_k^\top S_k^{-1}, \quad
\hat{x}_k = \hat{x}_{k|k-1} + K_k(y_k - H_k \hat{x}_{k|k-1}), \quad
P_k = (I - K_k H_k) P_{k|k-1}$$

In [ ]:
def kf_f(zk, uk, yk, Fk, Bk, Hk, Qk, Rk):
    """Discrete Kalman predict-update. Returns updated (x_est, P)."""
    x, P = zk
    # Predict
    x_pred = Fk @ x + Bk * uk
    P_pred = Fk @ P @ Fk.T + Qk
    # Update
    yk = np.atleast_1d(yk)
    y_res = yk - Hk @ x_pred
    S = Hk @ P_pred @ Hk.T + Rk
    K = P_pred @ Hk.T @ np.linalg.inv(S)
    x_upd = x_pred + K @ y_res
    P_upd = (np.eye(len(x)) - K @ Hk) @ P_pred
    return (x_upd, P_upd)

def kf_h(zk, **kwargs):
    return zk[0]  # state estimate

kf_block = DynamicalSystem(f=kf_f, h=kf_h)

## Parameters

In [ ]:
rng = np.random.default_rng(seed=42)

dt = 1.0
sim_time = np.arange(0, 60, dt)

# Reference
A_ref, T_ref = 30.0, 60.0

# PID
KP, KI, KD = 500.0, 30.0, 10.0

# Car
m, b = 1500.0, 50.0

# Kalman filter matrices
Fk = np.array([[1.0, dt],
               [0.0, 1.0 - (b / m) * dt]])
Bk = np.array([0.0, dt / m])   # control-input vector
Hk = np.array([[0.0, 1.0]])    # observe velocity
Qk = np.array([[0.05, 0.0],
               [0.0,  0.1]])
Rk = np.array([[2.0]])

# Measurement noise std for synthetic observations
v_noise_std = np.sqrt(Rk[0, 0])

## Simulation Loop — Figure 1

In [ ]:
# Initial states
rk = 0.0
ck = (0.0, 0.0)                          # PID: (e_prev, e_int)
pk = np.array([0.0, 0.0])               # plant: [position, velocity]
zk = (np.array([0.0, 0.0]), np.eye(2))  # KF: (x_est, P)

ref_out, ctrl_out, plant_out, kf_out = [], [], [], []

for tk in sim_time:

    # ── Reference ──────────────────────────────────────────────
    rk_next, _ = ref_block.step(rk=rk, tk=tk, A=A_ref, T=T_ref)
    ref_out.append(rk_next)

    # ── Controller (uses KF velocity estimate) ─────────────────
    v_est = zk[0][1]
    ck_next, uk = pid_block.step(ck=ck, rk=rk_next, xhatk=v_est,
                                  KP=KP, KI=KI, KD=KD, dt=dt)
    ctrl_out.append(uk)

    # ── Plant ──────────────────────────────────────────────────
    pk_next, vk = plant_block.step(pk=pk, uk=uk, m=m, b=b, dt=dt)
    plant_out.append(vk)

    # ── Kalman Filter ──────────────────────────────────────────
    yk_noisy = vk + rng.normal(0.0, v_noise_std)
    zk_next, _ = kf_block.step(zk=zk, uk=uk, yk=np.array([yk_noisy]),
                                Fk=Fk, Bk=Bk, Hk=Hk, Qk=Qk, Rk=Rk)
    kf_out.append(zk_next[0][1])  # estimated velocity (post-update)

    # ── Advance ────────────────────────────────────────────────
    rk, ck, pk, zk = rk_next, ck_next, pk_next, zk_next

In [ ]:
fig, axs = plt.subplots(4, 1, sharex=True, figsize=(10, 10))

axs[0].plot(sim_time, ref_out,   color="orange")
axs[0].set_ylabel("Reference (m/s)")
axs[0].set_title("Reference Output ($r_k$)")

axs[1].plot(sim_time, ctrl_out,  color="red")
axs[1].set_ylabel("Control Input (N)")
axs[1].set_title("PID Output ($u_k$)")
axs[1].ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

axs[2].plot(sim_time, plant_out, color="green")
axs[2].set_ylabel("Velocity (m/s)")
axs[2].set_title("Car Output ($y_k$)")

axs[3].plot(sim_time, kf_out,    color="blue")
axs[3].set_ylabel("Velocity (m/s)")
axs[3].set_xlabel("Time (s)")
axs[3].set_title("Kalman Filter Output ($\\hat{x}_k$)")

plt.tight_layout()
plt.savefig("car_cruise_control_figure.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Figure 2 — Two-System Comparison

The same framework applied to a second system: a **drone** controlled by an **LQR**
with state estimated by an **Unscented Kalman Filter (UKF)**.
Both systems track the same half-sine reference.

The LQR gain is computed offline via the discrete algebraic Riccati equation (DARE).
The UKF propagates sigma points through the (possibly nonlinear) dynamics.

> **Note:** This section requires `scipy` (`pip install scipy`).

In [ ]:
from scipy.linalg import solve_discrete_are

def lqr_gain(A, B, Q, R):
    """Compute discrete LQR gain K via the discrete algebraic Riccati equation."""
    P = solve_discrete_are(A, B, Q, R)
    return np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A

In [ ]:
def ukf_step(xhat, P, uk, yk, f_func, h_func, Q, R,
             alpha=1e-3, beta=2.0, kappa=0.0):
    """Unscented Kalman filter predict-update step."""
    n = len(xhat)
    lam = alpha**2 * (n + kappa) - n

    sqrtP  = np.linalg.cholesky((n + lam) * P)
    sigmas = np.column_stack(
        [xhat]
        + [xhat + sqrtP[:, i] for i in range(n)]
        + [xhat - sqrtP[:, i] for i in range(n)]
    )

    Wm = np.full(2 * n + 1, 1.0 / (2 * (n + lam)))
    Wm[0] = lam / (n + lam)
    Wc = Wm.copy()
    Wc[0] += 1 - alpha**2 + beta

    # Predict
    sf    = np.column_stack([f_func(sigmas[:, i], uk) for i in range(2 * n + 1)])
    xp    = sf @ Wm
    Pp    = sum(Wc[i] * np.outer(sf[:, i] - xp, sf[:, i] - xp)
                for i in range(2 * n + 1)) + Q

    # Update
    yk    = np.atleast_1d(yk)
    sh    = np.column_stack([h_func(sf[:, i]) for i in range(2 * n + 1)])
    yp    = sh @ Wm
    S     = sum(Wc[i] * np.outer(sh[:, i] - yp, sh[:, i] - yp)
                for i in range(2 * n + 1)) + R
    Pxy   = sum(Wc[i] * np.outer(sf[:, i] - xp, sh[:, i] - yp)
                for i in range(2 * n + 1))
    K     = Pxy @ np.linalg.inv(S)
    return xp + K @ (yk - yp), Pp - K @ S @ K.T

In [ ]:
# ── Drone parameters ──────────────────────────────────────────────
m_d, b_d = 1.0, 0.0   # 1 kg, no aerodynamic drag

A_d = np.array([[1.0, dt],
                [0.0, 1.0 - (b_d / m_d) * dt]])
B_d = np.array([[0.0], [dt / m_d]])

Q_lqr = np.diag([0.0, 100.0])   # penalise velocity error only
R_lqr = np.array([[1e-4]])
K_lqr = lqr_gain(A_d, B_d, Q_lqr, R_lqr)

Qk_d = np.array([[0.05, 0.0], [0.0, 0.1]])
Rk_d = np.array([[2.0]])

def drone_f(x, uk):
    uk_s = float(np.atleast_1d(uk)[0])
    return np.array([x[0] + dt * x[1],
                     x[1] + dt * (-b_d / m_d * x[1] + uk_s / m_d)])

def drone_h_obs(x):
    return np.array([x[1]])   # observe velocity

In [ ]:
# ── Initial conditions ────────────────────────────────────────────
rk1   = 0.0
ck1   = (0.0, 0.0)
pk1   = np.array([0.0, 0.0])
zk1   = (np.array([0.0, 0.0]), np.eye(2))

pk2   = np.array([0.0, 0.0])
xhat2 = np.zeros(2)
P2    = np.eye(2)

ref2_out, sys1_out, sys2_out = [], [], []

for tk in sim_time:

    # Reference
    rk1, _ = ref_block.step(rk=rk1, tk=tk, A=A_ref, T=T_ref)
    ref2_out.append(rk1)

    # ── System 1: Car + PID + KF ─────────────────────────────
    ck1_next, uk1 = pid_block.step(ck=ck1, rk=rk1, xhatk=zk1[0][1],
                                    KP=KP, KI=KI, KD=KD, dt=dt)
    pk1_next, vk1 = plant_block.step(pk=pk1, uk=uk1, m=m, b=b, dt=dt)
    zk1_next, _   = kf_block.step(
        zk=zk1, uk=uk1, yk=np.array([vk1 + rng.normal(0.0, v_noise_std)]),
        Fk=Fk, Bk=Bk, Hk=Hk, Qk=Qk, Rk=Rk
    )
    sys1_out.append(zk1_next[0][1])
    ck1, pk1, zk1 = ck1_next, pk1_next, zk1_next

    # ── System 2: Drone + LQR + UKF ──────────────────────────
    x_ref2 = np.array([pk2[0], rk1])          # track velocity, hold position
    uk2    = float(-K_lqr @ (xhat2 - x_ref2))
    pk2_next, vk2 = plant_block.step(pk=pk2, uk=uk2, m=m_d, b=b_d, dt=dt)
    xhat2_next, P2_next = ukf_step(
        xhat2, P2, uk2,
        np.array([vk2 + rng.normal(0.0, v_noise_std)]),
        drone_f, drone_h_obs, Qk_d, Rk_d
    )
    sys2_out.append(xhat2_next[1])
    pk2, xhat2, P2 = pk2_next, xhat2_next, P2_next

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 5))

ax2.plot(sim_time, ref2_out, "--", color="orange", linewidth=1.5, label="Reference (m/s)")
ax2.plot(sim_time, sys1_out,       color="blue",   linewidth=1.5, label="Car — PID + KF")
ax2.plot(sim_time, sys2_out,       color="purple", linewidth=1.5, label="Drone — LQR + UKF")

ax2.set_ylabel("Velocity (m/s)")
ax2.set_xlabel("Time (s)")
ax2.set_title("Two-System Comparison")
ax2.legend()

plt.tight_layout()
plt.savefig("two_system_comparison.pdf", format="pdf", bbox_inches="tight")
plt.show()